# Lesson 6.4 — Unit 6 Recap: Prediction with the Twin

Run-ahead + what-if + calibrated confidence. Now and next from one model.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, model_perception, understand,
                                          run_pipeline)
from modules.module10.twin import (DigitalTwin, GroundTruth, outcome_gap,
                                   monitor, predict, compare_futures)
def victim_and_xy(seed=1):
    w = GreenhouseWorld.demo_row(n=6, seed=seed)
    dets = model_perception(w, rng=np.random.default_rng(7))
    v = understand(dets, w)[1]['id']
    vxy = next(d['xy'] for d in dets if d['id']==v)
    return v, vxy
checks = []
v, vxy = victim_and_xy()
eff = {v: dict(obstacle=(vxy,0.25))}
ground = GroundTruth(GreenhouseWorld.demo_row(n=6, seed=1))
twin = DigitalTwin(ground.world); twin.sync(ground.report())
# run-ahead reproducible + reality untouched
before = [f.picked for f in ground.world.fruit]
a = predict(twin, rng_seed=7); b = predict(twin, rng_seed=7)
checks.append(a['harvested']==b['harvested'])
checks.append([f.picked for f in ground.world.fruit] == before)
# what-if futures differ on the affected fruit
fut = compare_futures(twin, {'nominal': None, 'obstacle': eff}, rng_seed=7)
checks.append(v in fut['nominal']['harvested'] and v in fut['obstacle']['skipped'])
# forecast inherits then sheds the gap via calibrate
g2 = GroundTruth(GreenhouseWorld.demo_row(n=6, seed=1), unmodeled=eff); t2 = DigitalTwin(g2.world)
checks.append(outcome_gap(predict(t2, sync_report=g2.report()), g2.run())['match'] is False)
t2.calibrate(eff)
checks.append(outcome_gap(predict(t2, sync_report=g2.report()), g2.run())['match'] is True)
print('runahead=%s untouched=%s whatif=%s gap_in=%s gap_out=%s' % tuple(checks))
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('INSTALLMENT C: the twin answers now (monitor) and next (predict) from one model. Next: Adapt.')
print('All checks passed.')


runahead=True untouched=True whatif=True gap_in=True gap_out=True
5/5 checks passed.
INSTALLMENT C: the twin answers now (monitor) and next (predict) from one model. Next: Adapt.
All checks passed.
